In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)



def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json


def generate_dataset():
    prompt = """You are generating synthetic test cases for evaluating an AI compliance pre-screening system for repo trading at an asset manager.

<system_being_tested>
The system receives a rule configuration, a current repo book, and a proposed new trade. It must determine whether adding the proposed trade breaches any parent rules, and whether PM override is required.

Rules are hierarchical:
- Parent rules contain child rules
- Each parent rule combines its children with AND or OR logic
- OR: parent breached if ANY child breached
- AND: parent breached only if ALL children breached
- PM override required if ANY parent rule is breached

Rule configurations vary per test case because they come from different client mandates.

All evaluations are net and post-trade: take the current book, apply the proposed trade, then check the resulting net position against each rule.
</system_being_tested>

<sign_convention>
- Positive notional = reverse repo (cash out to counterparty, collateral in)
- Negative notional = repo (cash in from counterparty, collateral out)
- Positions in the same maturity window offset each other in net calculations
</sign_convention>

<rule_dimensions>
Child rules aggregate on one of two dimensions:
maturity_month - aggregate net notional across all positions maturing in a given month, check against a numeric range (lower_bound, upper_bound)
counterparty - aggregate net notional across all positions with a given counterparty, check against a numeric range (lower_bound, upper_bound)
</rule_dimensions>

<task>
Generate diverse test cases. Each test case contains:
1. A rule configuration (one or more parent rules with children and AND/OR logic)
2. A current repo book (3-8 positions with signed notionals)
3. A proposed new trade (signed notional)
4. The expected output after applying the rules

Vary scenarios to cover:
- Clean pass (no parent rules breached)
- OR parent breached (at least one child triggers)
- AND parent breached (all children trigger)
- AND parent NOT breached despite partial child breach (critical edge case)
- Multiple parent rules breached simultaneously
- Net calculation reduces exposure (proposed trade offsets existing position)
- Net calculation increases exposure beyond limit
- Edge cases at exactly the threshold
</task>

<output_format>
Return a JSON array. Each element:
{
  "scenario_label": "short description of what this case tests",
  "task": {
    "rules": [
      {
        "parent_rule_id": "MATURITY_CONCENTRATION" or "COUNTERPARTY_EXPOSURE",
        "logic": "AND" or "OR",
        "children": [
          {
            "id": "string identifier",
            "description": "human readable rule",
            "dimension": "maturity_month" or "counterparty",
            "filter": "YYYY-MM for maturity_month, counterparty name for counterparty",
            "lower_bound": integer (GBP, can be negative),
            "upper_bound": integer (GBP, can be negative)
          }
        ]
      }
    ],
    "current_book": [
      {"counterparty": "string", "notional": integer (signed), "maturity": "YYYY-MM-DD"}
    ],
    "proposed_trade": {
      "counterparty": "string", "notional": integer (signed), "maturity": "YYYY-MM-DD"
    }
  },
  "expected": {
    "parent_results": [
      {
        "parent_rule_id": "string",
        "logic": "AND" or "OR",
        "breached": boolean,
        "child_results": [
          {
            "id": "string",
            "breached": boolean,
            "net_value": integer (calculated net for that dimension/filter post-trade),
            "lower_bound": integer,
            "upper_bound": integer
          }
        ],
        "reasoning": "explanation referencing specific numbers"
      }
    ],
    "pm_override_required": boolean,
    "verdict": "string describing overall outcome"
  }
}
</output_format>

<constraints>
- All net calculations must be mathematically correct based on the book + proposed trade
- A child rule breaches when net_value falls OUTSIDE the [lower_bound, upper_bound] range
- Counterparty names from: Barclays, JPM, Goldman Sachs, Morgan Stanley, Citi, HSBC, BNP Paribas, Deutsche Bank
- Maturity dates between 2026-06-15 and 2026-12-31
- Notionals between -800000000 and +800000000 (signed)
- pm_override_required is true if and only if any parent_results entry has breached=true
- For AND logic scenarios, deliberately include cases where some but not all children breach
- Reasoning must reference specific calculated values and limits
</constraints>

Generate 20 test cases covering the full range of scenarios.
Distribution:
- 3 clean pass
- 3 OR parent breached
- 3 AND parent breached
- 3 AND parent NOT breached despite partial child breach
- 3 multiple parents breached simultaneously
- 3 trade reduces net exposure (offset scenarios)
- 2 edge cases at exactly the threshold

Vary the rule configurations - different parents, different children, different AND/OR logic, different bounds across cases."""


    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [4]:
dataset = generate_dataset()


with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

JSONDecodeError: Expecting value: line 99 column 44 (char 3220)

In [41]:
def run_prompt(test_case):
    #combines boths the prompts and the test cases and provide these as messages to claude
    prompt = f"""
    Please solve the following task:

    {test_case["task"]}

    * Respond only with Python, JSON, or a plain Regex
    * Do not add any comments or commentary or explanation
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output



In [42]:
#pases in the test cases generated by a model, and the ouptuts to those test cases
def grade_by_model(test_case, output):
       # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    Solution Criteria: {test_case["solution_criteria"]}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [43]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [44]:
def run_test_case(test_case):
    #returns json with the test case, and output from claude with the score
    output = run_prompt(test_case)

    #todo -grading
    model_grade = grade_by_model(test_case,output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    #score = 10

    syntax_score = grade_syntax(output, test_case)
    score = (model_score+syntax_score)/2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [45]:
def run_eval(test_case):
    results =  []

    total = 0
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

        total+=result["score"]
    avg = total/len(dataset)
    print(avg)


    average = []
    return results

In [46]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset)

8.166666666666666


In [ ]:
print(json.dumps(results, indent = 2))

[
  {
    "output": "\nimport re\n\ndef extract_account_id(arn):\n    match = re.search(r':(\\d{12}):', arn)\n    return match.group(1) if match else None\n",
    "test_case": {
      "task": "Extract the AWS account ID from an ARN (Amazon Resource Name) string",
      "format": "regex",
      "solution_criteria": "The regex should match and capture the 12-digit account ID from ARNs in the format arn:aws:service:region:account-id:resource"
    },
    "score": 8.0,
    "reasoning": "The solution works for well-formed, standard ARNs but is fragile. It would incorrectly extract digits from malformed ARNs (e.g., 'arn:aws:service:1234567890123:account-id:resource' would match the wrong field). The regex lacks context awareness of ARN structure. A more robust solution would validate the full ARN format before extraction, or at minimum verify the digit group is in the correct position (5th colon-separated field). The code lacks error handling documentation and type hints."
  },
  {
    "outpu

: 